In [1]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# function to fill nas in both categorical and numerical columns
def fill_nas(data_to_change, data_to_read):
    cat_cols = data_to_change.select_dtypes(include="object").columns

    for col in cat_cols:
        data_to_change[col]=data_to_change[col].fillna(data_to_read[col].mode()[0])
    #now numerical columns
    num_cols = data_to_change.select_dtypes(exclude="object").columns
    for col in num_cols:
        data_to_change[col] = data_to_change[col].fillna(data_to_read[col].mean())
    return data_to_change
# func to turn categorical columns to numerical using mean encoding based on SalePrice mean
def cat_to_num_func(data_to_change, data_for_mapping):
    global_mean = data_for_mapping["SalePrice"].mean()
    cat_to_num_maps = {}
    cat_cols = data_to_change.select_dtypes(include="object").columns
    for col in cat_cols:
        cat_to_num_maps[col] = data_for_mapping.groupby(col)["SalePrice"].mean()


    for col in cat_cols:
        data_to_change[col + "_num"] = data_to_change[col].map(cat_to_num_maps[col]).fillna(global_mean)
        data_to_change.drop(col, axis=1, inplace=True)  
    return data_to_change

# func to scale data
def data_scale(data_to_change, scaler=None):
    if scaler is None:
        scaler = StandardScaler()
        X_scaled = scaler.fit_transform(data_to_change)
    else:
        X_scaled = scaler.transform(data_to_change)
    return X_scaled, scaler
def preprocess (data_to_change, data_for_mapping, scaler=None):
    data_to_change=pd.DataFrame(data_to_change.copy())
    data_for_mapping=pd.DataFrame(data_for_mapping.copy())
    data_to_change = fill_nas(data_to_change, data_for_mapping)
    data_to_change = cat_to_num_func(data_to_change, data_for_mapping)

    # drop Id and SalePrice if they exist
    for col in ["Id", "SalePrice"]:
        if col in data_to_change.columns:
            data_to_change.drop(columns=[col], inplace=True)
    col_names = data_to_change.columns.tolist()
    X_scaled, scaler = data_scale(data_to_change, scaler)
    return pd.DataFrame(X_scaled, columns=col_names), scaler

from sklearn.feature_selection import RFE
from sklearn.linear_model import LinearRegression

  
# func to do RFE and return only selected number of features
def data_rfe(data_to_change, data_to_fit, y, number_of_features=20):
    data_to_fit = pd.DataFrame(data_to_fit)
    data_to_change = pd.DataFrame(data_to_change)
    modelLR = LinearRegression()
    rfe=RFE(modelLR, n_features_to_select=number_of_features, step=0.1)
    rfe.fit(data_to_fit, y)
    
    rfe_selected_features = data_to_fit.columns[rfe.support_].to_list()
    #for i, feature in enumerate(rfe_selected_features, 1):
        #print(f"{i}. {feature}")
    data_to_change_rfe = data_to_change[rfe_selected_features]
    return data_to_change_rfe





In [2]:
import mlflow
import mlflow.sklearn
import pandas as pd
import numpy as np

mlflow.set_tracking_uri("https://dagshub.com/slomi23/slomi23_ML_Assignment_1.mlflow")
mlflow.set_experiment("House Price Prediction")

<Experiment: artifact_location='mlflow-artifacts:/a217c730665f40dbb8acf21d888da7f8', creation_time=1775886387111, experiment_id='0', last_update_time=1775886387111, lifecycle_stage='active', name='House Price Prediction', tags={'mlflow.experimentKind': 'custom_model_development'}, trace_location=None, workspace='default'>

In [3]:
test = pd.read_csv("house-prices-advanced-regression-techniques/test.csv")

runs = mlflow.search_runs(experiment_names=["House Price Prediction"])
best_run = runs[runs["tags.mlflow.runName"] == "linear_regression_without_OHE_and_kfold_final"].iloc[0]
print(f"Val RMSE mean: {best_run['metrics.val_rmse']}")
#print(runs["tags.mlflow.runName"].tolist())


Val RMSE mean: 0.14338270136704678


In [4]:
client = mlflow.tracking.MlflowClient()
artifacts = client.list_artifacts("021ad6ce146343d88dd824bcf13dab38")
for a in artifacts:
    print(a.path)
print(best_run['run_id'])

b532b276ae854fc387e6c7af496a32ac


In [5]:
import os
os.environ["KAGGLE_USERNAME"] = "slomi23"
os.environ["KAGGLE_KEY"] = "KGAT_cd8dfb20444563d3270ec0174f8df58f"

In [6]:
!pip install kaggle --quiet

# load retrained model (version 4 trained on log target)
loaded_model = mlflow.sklearn.load_model("models:/linear_regression_without_OHE_and_kfold_final/3")

X_train_raw = pd.read_csv("house-prices-advanced-regression-techniques/train.csv")

X_train_processed, scaler = preprocess(X_train_raw, X_train_raw)
X_test_processed, _ = preprocess(test, X_train_raw, scaler)

selected_cols = pd.read_csv("selected_features.csv").squeeze().tolist()
X_test_selected = X_test_processed[selected_cols]

preds = np.expm1(loaded_model.predict(X_test_selected))
submission = pd.DataFrame({"Id": test["Id"], "SalePrice": preds})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print(submission.shape)
print(submission["SalePrice"].describe())

     Id      SalePrice
0  1461  100901.201990
1  1462  136782.749319
2  1463  164501.349773
3  1464  177814.154603
4  1465  198525.110180
(1459, 2)
count      1459.000000
mean     177823.196319
std       77572.425939
min       56295.352761
25%      126591.813883
50%      158263.656310
75%      207365.616249
max      748254.406661
Name: SalePrice, dtype: float64
